# The heterodyne (relative-binning) likelihood

This notebook walks through HyperWave's fastest likelihood. The idea
(Zackay, Dai & Venumadhav 2018): near the posterior peak every waveform is
*almost* the reference waveform $h_0$, so the smooth ratio $r(f)=h/h_0$ can be
approximated as piecewise-linear over a handful of frequency bins. The data
enter only through per-bin summary products, and each likelihood evaluation
needs the waveform at the **bin edges only** — a few hundred frequencies
instead of $10^4$–$10^6$ grid points.


In [ ]:
import numpy as np, matplotlib.pyplot as plt
from hyperwave.detectors.lvk import GW, DetectorNoise
from hyperwave.likelihoods import GWLikelihoods, HeterodyneLikelihood, heterodyne_bin_edges


## 1. Build a BBH injection (Gaussian noise, H1+L1)


In [ ]:
TRIGGER = 1268189526.951953
NAMES = ['chirp_mass', 'mass_ratio', 'luminosity_distance', 'phase']
STATIC = dict(psi=1.1, ra=1.375, dec=-0.2108, chi_1=0.0, chi_2=0.0,
              cos_theta_jn=np.cos(0.4), cos_tilt_1=1.0, cos_tilt_2=1.0,
              phi_12=0.0, phi_jl=0.0, geocent_time=TRIGGER)
m1, m2 = 36.0, 29.0
theta_ref = np.array([(m1+m2)*(m1*m2/(m1+m2)**2)**0.6, m2/m1, 600.0, 0.9])

noise = DetectorNoise(4.0, 2048.0, TRIGGER, ['H1','L1'], minimum_frequency=20.0, maximum_frequency=512.0)
noise.generate_noise(real_noise=False, seed=42)
template = GW(noise, approximant='IMRPhenomPv2', parameters=NAMES, static_parameters=STATIC)
template.make_injections_to_ifo(theta_ref)
f, asd0 = template.detector_asd_masked(0)
psd = np.array([asd0**2, template.detector_asd_masked(1)[1]**2])
data = np.array([template.detector_data_fd(0), template.detector_data_fd(1)])
f.size


## 2. The bins

Edges are placed so a PN-like phase can evolve by at most `eps` radians per
bin — dense at low frequency where the phase runs fastest:


In [ ]:
edges = heterodyne_bin_edges(f, eps=0.1)
print(f'{f.size} grid points -> {edges.size} edges')
plt.figure(figsize=(7,2.4))
plt.plot(f[edges], np.zeros_like(edges), '|', ms=18)
plt.xscale('log'); plt.xlabel('frequency [Hz]'); plt.yticks([]); plt.title('bin edges')


## 3. Build the likelihood and check it against the exact one

`from_lvk_template` evaluates $h_0$ once on the full grid and builds a second
LAL template in *sequence mode* on the sparse edge grid. The dropped
$\langle d|d\rangle/2$ constant cancels in logL **differences**:


In [ ]:
exact = GWLikelihoods(data=data, f=f, ifos_list=['H1','L1'], noise=psd,
                      template=template, ddims=False, nsegs=1)
het = HeterodyneLikelihood.from_lvk_template(template, data=data, f=f, psd=psd,
                                             ifos_list=['H1','L1'], theta_ref=theta_ref, eps=0.1)
rng = np.random.default_rng(7)
trials = np.tile(theta_ref, (24,1))
trials[:,0] += rng.uniform(-0.02, 0.02, 24)      # chirp mass
trials[:,2] *= 1 + rng.uniform(-0.05, 0.05, 24)  # distance
g = np.atleast_1d(exact.gaussian(trials)) - float(exact.gaussian(theta_ref))
h = np.atleast_1d(het.logl(trials))    - float(het.logl(theta_ref))
plt.figure(figsize=(4.2,4)); plt.plot(g, h, '.'); lim=[g.min(), g.max()]
plt.plot(lim, lim, 'k--', lw=0.8); plt.xlabel('exact $\\Delta\\ln L$'); plt.ylabel('heterodyne $\\Delta\\ln L$')
print('max |error| =', float(np.max(np.abs(h-g))))


## 4. Speed


In [ ]:
import time
batch = np.tile(theta_ref, (64,1)); batch[:,0] += rng.uniform(-0.02,0.02,64)
exact.gaussian(batch[:2]); het.logl(batch[:2])   # warm-up
t0=time.perf_counter(); exact.gaussian(batch); tf=time.perf_counter()-t0
t0=time.perf_counter(); het.logl(batch);     th=time.perf_counter()-t0
print(f'full       : {tf/64*1e3:6.2f} ms/eval')
print(f'heterodyne : {th/64*1e3:6.2f} ms/eval   ({tf/th:.1f}x)')


The per-evaluation cost of the heterodyne likelihood is set by the number of
*bins*, not the grid — re-run this notebook with `duration=64.0` in the
`DetectorNoise` line and the speedup grows to ~70x while the heterodyne
time stays ~0.3 ms.

## 5. PE

For a full sampled run see `examples/heterodyne/bbh_heterodyne_pe.py`
(or submit `examples/clusters/heterodyne_pe.slurm`): the heterodyne logL
plugs into `LVKinference` exactly like any other HyperWave likelihood.
